# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided exploration for loading and investigating the FAIR^2 CROISSANT dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")

# Show main metadata attributes
# (see all available attributes with dir(metadata))
print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")

## 2. Data Overview
Review available record sets and their field and column `@id`s.

Let's list all record sets defined in the metadata (each with its `@id`) and inspect their fields and columns, as per Croissant best practice.

In [ ]:
# Get all record sets (as RecordSet objects)
record_sets = list(metadata.record_sets)
print(f"Record Sets in the dataset ({len(record_sets)} found):")
for rs in record_sets:
    print(f"  - RecordSet name: {rs.name}, @id: {rs.id}")
    # List fields in the record set
    if rs.fields:
        print("    Fields:")
        for f in rs.fields:
            print(f"      - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    # List columns if available
    if rs.columns:
        print("    Columns:")
        for c in rs.columns:
            print(f"      - {c.name} (@id: {c.id})")

## 3. Data Extraction
We'll extract data from each record set using their `@id` values as reference. Data from each record set will be loaded into a pandas DataFrame for downstream analysis.

In [ ]:
# Prepare a list of record set @id strings
record_set_ids = [rs.id for rs in record_sets]

# Read the records of each set to separate DataFrames
dataframes = dict()
for recset_id in record_set_ids:
    print(f"Loading records for RecordSet: {recset_id}")
    try:
        records = list(dataset.records(record_set=recset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[recset_id] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(f"  Number of records: {len(df)}\n")
        else:
            print("  No records found.\n")
    except Exception as e:
        print(f"  Failed to load records: {e}\n")

# Preview the first record set (if available)
if record_set_ids:
    first_rs = record_set_ids[0]
    if first_rs in dataframes:
        print(f"Preview of first record set `{first_rs}`:")
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (`@id`) from a record set for filtering, normalization, and grouping. 

You can modify the selected record set and field by editing the variables below.

In [ ]:
# --- User Editable Section ---
# Set these to an existing record set @id and a numeric field @id from the earlier overview
selected_record_set_id = record_set_ids[0] if record_set_ids else None  # Pick the first record set by default

# Identify a numeric field from the selected record set
rs_obj = next((rs for rs in record_sets if rs.id == selected_record_set_id), None)
numeric_field_id = None
# Heuristic: Pick the first field/column with numeric datatype
if rs_obj and rs_obj.fields:
    for f in rs_obj.fields:
        if f.data_type and f.data_type.lower() in ('number', 'float', 'integer'):
            numeric_field_id = f.id
            break
if numeric_field_id is None:
    # Try columns
    if rs_obj and rs_obj.columns:
        for c in rs_obj.columns:
            if hasattr(c, 'data_type') and c.data_type and c.data_type.lower() in ('number', 'float', 'integer'):
                numeric_field_id = c.id
                break

print(f"Working with RecordSet: {selected_record_set_id}\nNumeric field: {numeric_field_id}")

if selected_record_set_id and numeric_field_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Try to get group field (a string categorical field)
    group_field_id = None
    if rs_obj and rs_obj.fields:
        for f in rs_obj.fields:
            if f.data_type and f.data_type.lower() == 'text' and f.id != numeric_field_id:
                group_field_id = f.id
                break
    # Filtering - handle parsing numbers from string if needed
    # Convert field to float if possible
    if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by group_field_id (if available)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No valid record set and numeric field found for EDA.")

## 5. Visualization
We can now visualize the distribution of the selected numeric field and its normalized values.

Let's plot histograms for the chosen field and the grouped means if applicable.

In [ ]:
import matplotlib.pyplot as plt

# Histogram for the numeric field
if selected_record_set_id and numeric_field_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8,4))
    # Drop NaN for plotting
    df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # Normalized field if present
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in df:
        plt.figure(figsize=(8,4))
        df[norm_col].dropna().hist(bins=30, color='orange')
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(norm_col)
        plt.ylabel("Count")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We have loaded the FAIR^2 mass spectrometry dataset via its Croissant schema, listed available record sets and their fields, loaded data into pandas, filtered and normalized a numeric field, and visualized its distribution.

This pipeline demonstrates best practices for programmatic interaction with CROISSANT datasets using unique `@id` references for entities such as record sets and fields. Modify or extend this notebook for further analysis of the dataset!